# 02 — Feature Engineering

Derive the 5 ML-ready features from cleaned keystroke data.

In [ ]:
import pandas as pd
import numpy as np

fixed = pd.read_csv('../data/cleaned/cleaned_data.csv')
freq  = pd.read_csv('../data/raw/frequency.csv', sep=';')
freq['TotTime'] = pd.to_numeric(freq['TotTime'], errors='coerce')
print(f'Loaded: fixed {fixed.shape}, freq {freq.shape}')

## IQR clipping helper

In [ ]:
def iqr_clip(s, factor=1.5):
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    return s.clip(Q1 - factor*(Q3-Q1), Q3 + factor*(Q3-Q1))

## Aggregate per user × emotion

In [ ]:
agg = fixed.groupby(['userId','emotionIndex']).agg(
    mean_dwell    = ('D1U1', 'mean'),
    _std_flight   = ('D1D2', 'std'),
    _mean_flight  = ('D1D2', 'mean'),
    median_flight = ('D1D2', 'median'),
    n_keys        = ('D1U1', 'count'),
).reset_index()

agg['cv_flight'] = agg['_std_flight'] / agg['_mean_flight']
agg.drop(columns=['_std_flight','_mean_flight'], inplace=True)
print(agg.head())

## Merge delete frequency and total time

In [ ]:
freq_agg = freq.groupby(['User ID','emotionIndex']).agg(
    mean_del_freq = ('delFreq', 'mean'),
    mean_tot_time = ('TotTime', 'mean'),
).reset_index().rename(columns={'User ID':'userId'})

features = agg.merge(freq_agg, on=['userId','emotionIndex'], how='left')
print(f'Feature matrix: {features.shape}')

## Apply IQR clipping

In [ ]:
for col in ['mean_dwell','median_flight','cv_flight']:
    features[col] = iqr_clip(features[col])
print('Clipping applied')

## Save

In [ ]:
features.to_csv('../data/processed/final_features.csv', index=False)
print('Saved: ../data/processed/final_features.csv')
print(features.describe().round(2))